### Import Library dan Membaca Dataset



In [10]:
import os
import pandas as pd
import numpy as np

# Membaca dataset NJOP
df_njop = pd.read_csv('../dataset/njop_jakarta_v2.csv')

# Membaca dataset harga rumah
df_house = pd.read_csv('../dataset/jakarta_house_v2.csv')

print("INFORMASI BEBERAPA BARIS PERTAMA DATASET njop_jakarta:")
display(df_njop.head())

print("\nINFORMASI BEBERAPA BARIS PERTAMA DATASET jakarta_house:")
display(df_house.head())

INFORMASI BEBERAPA BARIS PERTAMA DATASET njop_jakarta:


,city,district,njop/m2
0,Jakarta Selatan,Kebayoran Lama,6343000
1,Jakarta Selatan,Kebayoran Lama,19843000
2,Jakarta Selatan,Kebayoran Lama,6343000
3,Jakarta Selatan,Kebayoran Lama,9063000
4,Jakarta Selatan,Kebayoran Lama,19843000



INFORMASI BEBERAPA BARIS PERTAMA DATASET jakarta_house:


,index,price,district,city,bed_rooms,bath_rooms,carport,land_area,building_area
0,1,5900000000,Kalideres,Jakarta Barat,2,4,2,250.0,350.0
1,2,2700000000,Grogol Petamburan,Jakarta Barat,4,2,0,100.0,225.0
2,3,2200000000,Grogol Petamburan,Jakarta Barat,3,3,0,60.0,140.0
3,4,1900000000,Grogol Petamburan,Jakarta Barat,3,2,0,60.0,120.0
4,5,2100000000,Grogol Petamburan,Jakarta Barat,4,3,0,56.0,108.0


### Standarisasi Data dan Koreksi Nama Wilayah

In [11]:
# Menghapus spasi di awal dan akhir teks
df_njop['city'] = df_njop['city'].str.strip()
df_njop['district'] = df_njop['district'].str.strip()
df_house['city'] = df_house['city'].str.strip()
df_house['district'] = df_house['district'].str.strip()


# Standarisasi nama 'Setiabudi' -> 'Setia Budi' agar sesuai dengan dataset NJOP
df_house['district'] = df_house['district'].replace({'Setiabudi': 'Setia Budi'})

# Membuat kamus referensi resmi district -> city langsung dari dataset NJOP
district_to_city = df_njop.drop_duplicates('district').set_index('district')['city'].to_dict()


# Mengoreksi kesalahan nama city pada data rumah sesuai referensi resmi
df_house['city'] = df_house['district'].map(district_to_city).fillna(df_house['city'])

print("Standarisasi dan koreksi nama wilayah berhasil dilakukan!")

Standarisasi dan koreksi nama wilayah berhasil dilakukan!


### Menghitung Median NJOP per m² Berdasarkan City dan District

In [12]:
# Melakukan agregasi: median njop/m2 dan jumlah data valid per kombinasi city dan district
df_njop_agg = df_njop.groupby(['city', 'district'], as_index=False).agg(
    **{
        'njop/m2': ('njop/m2', 'median')
    }
)

print(f"Jumlah kombinasi city dan district: {len(df_njop_agg)}")

Jumlah kombinasi city dan district: 42


### Menampilkan Hasil Perhitungan Median



In [13]:
# Mengurutkan hasil berdasarkan city kemudian district secara alfabetis (A-Z)
df_njop_final = df_njop_agg.sort_values(by=['city', 'district']).reset_index(drop=True)

print(f"TABEL HASIL PERHITUNGAN MEDIAN NJOP PER M2")
display(df_njop_final)

TABEL HASIL PERHITUNGAN MEDIAN NJOP PER M2


,city,district,njop/m2
0,Jakarta Barat,Cengkareng,4605000.0
1,Jakarta Barat,Grogol Petamburan,11305000.0
2,Jakarta Barat,Kalideres,3843000.0
3,Jakarta Barat,Kebon Jeruk,6195000.0
4,Jakarta Barat,Kembangan,6195000.0
5,Jakarta Barat,Palmerah,6963000.0
6,Jakarta Barat,Taman Sari,13125000.0
7,Jakarta Barat,Tambora,10455000.0
8,Jakarta Pusat,Cempaka Putih,7455000.0
9,Jakarta Pusat,Gambir,23295000.0


### Menggabungkan Dengan Dataset jakarta_house

In [14]:
# Menggabungkan dataset rumah dengan median NJOP
df_house_njop = df_house.merge(
    df_njop_final,
    on=['city', 'district'],
    how='left',
    validate='many_to_one'
)

### Memeriksa Jumlah Data

In [15]:
print(f'Jumlah data sebelum merge: {len(df_house)}')
print(f'Jumlah data setelah merge: {len(df_house_njop)}')

Jumlah data sebelum merge: 9533
Jumlah data setelah merge: 9533


### Memeriksa Missing Value

In [16]:
print(
    'Missing value NJOP:',
    df_house_njop['njop/m2'].isnull().sum()
)

Missing value NJOP: 0


### Memeriksa Persentase Data Yang Berhasil Mendapatkan NJOP

In [17]:
# Menghitung persentase data yang berhasil mendapatkan NJOP
persentase_njop = (
    df_house_njop['njop/m2'].notnull().mean() * 100
)

print(f'Persentase data dengan NJOP: {persentase_njop:.2f}%')

Persentase data dengan NJOP: 100.00%


### Menyimpan Dataset

In [18]:
df_house_njop.to_csv('../dataset/jakarta_house_v3.csv',index=False)


print("Dataset berhasil disimpan di folder dataset dengan nama jakarta_house_v3.csv")

Dataset berhasil disimpan di folder dataset dengan nama jakarta_house_v3.csv
